In [1]:
import pandas as pd
import dask.dataframe as dd 
import numpy as np
import datetime as dt
from datetime import date
pd.set_option('display.max_columns', None)

In [2]:
### Facility CBSA codes 
### Author: Nadia Ghazali 
### Description: The purpose of this script is to create a crosswalk of area wage indexes for each nursing facility in 2019 to use with the 2019 TAF long term care claims. 
### To do so, I used a series of files to crosswalk between facility NPI (the identifier in the TAF claims) and CBSA (Core Based Statistical Area). 
### Urban area index is assigned based on CBSA, while rural area wage index is the same for all counties within a given state. 

### I use the following files to crosswalk between facility NPI and CBSA code:
### NPI XWALK: facility NPI -> zip code 
### PROVIDER INFO: zip code -> county SSA code 
### CBSA FIPS XW: county SSA code -> CBSA code
### URBAN WAGE INDEX: CBSA code -> area wage index 



In [4]:
def clean_npi_xwalk(): 
    """ Read in and clean csv file which crosswalks between facility NPI and zip code """ 
    # read in NPI crosswalk 
    npi_xwalk = pd.read_csv('/gpfs/data/cms-share/duas/56930/Joe/bed_blocking/NPI/npidata.csv',header=0, 
                            usecols=['NPI','Entity Type Code', 'Provider Business Practice Location Address Postal Code'], 
                            dtype={'NPI':'string','Entity Type Code':'string','Provider Business Practice Location Address Postal Code': 'string'})
    print(npi_xwalk.columns)
    
    npi_xwalk = npi_xwalk.rename(columns={'Entity Type Code':'Entity_Type_Code', 'Provider Business Practice Location Address Postal Code':'Practice_Zip'})
                                          
    # keep 'organization' and drop 'individual' entity types 
    npi_xwalk = npi_xwalk.loc[npi_xwalk['Entity_Type_Code']=='2']
    
    # format zip codes 
    npi_xwalk['Practice_Zip'] = npi_xwalk['Practice_Zip'].astype(str).str[:5]

    return npi_xwalk


Index(['NPI', 'Entity Type Code',
       'Provider Business Practice Location Address Postal Code'],
      dtype='object')


In [3]:
def list_all_states_npis(year, filetype, state_list):
    
    """ Reads in long term claims file to generate a list of all facility NPIs """ 
    state_dataframes = []

    if filetype == 'max':

        npi_col = ['NPI']
    else: 
        npi_col = ['facility_npi']

    for state in state_list: 
            claims = pd.read_parquet(f'/{directory_to_store_cleaned_lt_claims_files}/{filetype}_lt_NF_claims/{year}/{state}', columns=npi_col)
            claims['STATE'] = f'{state}'
            
            # drop duplicates
            claims = claims.drop_duplicates(subset=npi_col, keep='first').reset_index(drop=True)

            state_dataframes.append(claims)
        
    
    all_states_claims_npis = pd.concat(state_dataframes)

    if npi_col == ['facility_npi']:
        all_states_claims_npis = all_states_claims_npis.rename(columns={'facility_npi':'NPI'})

    return all_states_claims_npis


In [5]:
def merge_claims_npi(all_states_claims_npis, npi_xwalk): 
    """ Merges list of all NPIs in claims with cleaned NPI xwalk" 
    # merge MAX/TAF claims with NPI crosswalk 
    claims_npi = all_states_claims_npis.merge(npi_xwalk, how='left', on='NPI', indicator='claims_npi_merge')
    claims_npi = claims_npi.loc[claims_npi['claims_npi_merge']=='both']
    
    print(claims_npi['claims_npi_merge'].value_counts())

    return claims_npi
    

In [6]:
def merge_provider_info(claims_npi): 
    """ Merges provider info dataset with list of NPIs in claims""" 

    provider_info = pd.read_csv(f'/{directory_to_store_provider_info_files}/ProviderInfo_Download.csv', 
                            usecols=['ZIP','STATE','COUNTY_SSA','PARTICIPATION_DATE','County_name'], header=0, dtype='str')

   
    provider_info = provider_info.drop_duplicates(subset=['ZIP','COUNTY_SSA'], keep='first')
    
    print(provider_info.columns.to_list())
    
    state_codes = pd.read_csv(f'{directory_to_store_state_SSA_codes}/state_SSA_codes.csv', dtype={'STATE_CD':'str', 'STATE_NAME':'str', 'STATE':'str'})
    
    provider_info = provider_info.merge(state_codes, how='left', on='STATE', indicator='state_code_merge')
    provider_info = provider_info.drop(columns=['STATE'])
    
    provider_info = provider_info.loc[provider_info['state_code_merge']=='both']
    provider_info = provider_info.drop(columns=['state_code_merge'])

    claims_npi_provider = claims_npi.merge(provider_info, how='left', left_on='Practice_Zip', right_on='ZIP', indicator='claims_provider_zip_merge')
    claims_npi_provider = claims_npi_provider.loc[claims_npi_provider['claims_provider_zip_merge']=='both']
    print(claims_npi_provider['claims_provider_zip_merge'].value_counts())
    return claims_npi_provider


In [7]:
def merge_facility_cbsa_codes(claims_npi_provider): 
    """ Adds CBSA codes to list of facility NPIS """ 
    county_ssa_cbsa = pd.read_csv(f'/{directory_to_store_ssa_county_to_cbsa_crosswalk}/ssa_fips_state_county2019.csv', header=0, dtype={'county':'str', 'state':'str', 'ssacd':'str', 'fipscounty':'str', 'cbsa':'str', 'cbsaname':'str'})
    county_ssa_cbsa['ssacd'] = county_ssa_cbsa['ssacd'].str[-3:]
    claims_npi_zip_county_cbsa_merge = claims_npi_provider.merge(county_ssa_cbsa, how='left', left_on=['COUNTY_SSA','STATE'], right_on=['ssacd','state'], indicator='county_ssa_merge')
    print(claims_npi_zip_county_cbsa_merge['county_ssa_merge'].value_counts())

    return claims_npi_zip_county_cbsa_merge

In [8]:
def merge_wage_indexes(claims_npi_zip_county_cbsa_merge, year): 
    """ Merges wage index dataframes with facility NPIs, with a specific index for each urban CBSA and a state-specific rural index for all  """ 
    urban_wage_index = pd.read_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/urban_cbsa_wage_index_{year}.csv', 
                                   dtype = 'str')
    urban_wage_index = urban_wage_index.rename(columns={'CBSA Code':'CBSA_Code','Wage Index':'Wage_Index'})

    cbsa_wage_index_merge = claims_npi_zip_county_cbsa_merge.merge(urban_wage_index, how='left', left_on='cbsa', right_on='CBSA_Code', indicator='urban_CBSA_merge')
    urban_cbsa_wage_index_merge = cbsa_wage_index_merge.loc[cbsa_wage_index_merge['urban_CBSA_merge']=='both']
    print('first merge: ' + str(cbsa_wage_index_merge['urban_CBSA_merge'].value_counts()))
    
    urban_cbsa_wage_index_merge['urban/rural'] = 'urban'

    rural_cbsa_wage_index_merge = cbsa_wage_index_merge.loc[cbsa_wage_index_merge['urban_CBSA_merge']=='left_only']
    rural_cbsa_wage_index_merge['urban/rural'] = 'rural'
    rural_cbsa_wage_index_merge = rural_cbsa_wage_index_merge.drop(columns=['Wage_Index'])
    print('second_merge: ' + str(rural_cbsa_wage_index_merge['urban_CBSA_merge'].value_counts()))
    
    # for failed merges, try to merge with rural area wage index file (merge based on state only) 
    rural_wage_index = pd.read_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/rural_wage_index_{year}.csv',
                                   usecols = ['Nonurban Area', 'STATE', 'Wage Index'],
                                   dtype='str')
    rural_wage_index = rural_wage_index.rename(columns={'Wage Index':'Wage_Index'})
    rural_cbsa_wage_index_merge = rural_cbsa_wage_index_merge.merge(rural_wage_index, how='left', left_on='STATE', right_on='STATE')
    facility_area_wage_index = pd.concat([urban_cbsa_wage_index_merge, rural_cbsa_wage_index_merge])
    facility_area_wage_index = facility_area_wage_index.sort_values(by=['STATE'])

    
    facility_area_wage_index['Wage_Index'] = facility_area_wage_index['Wage_Index'].replace('-----', np.nan)
    facility_area_wage_index['Wage_Index'] = facility_area_wage_index['Wage_Index'].astype(float)
    facility_area_wage_index['facility_npi'] = facility_area_wage_index['NPI'].astype('str')

    return facility_area_wage_index


In [2]:

def prep_area_wage_file(year, filetype, state_list): 
    """ Uses the previous functions to prepare and read out annual area wage csv file, which provides an area wage index for each facility """ 
    print(f'***YEAR: {year}')

    all_states_claims_npis = list_all_states_npis(year, filetype, state_list)
    npi_xwalk = clean_npi_xwalk()
    claims_npi = merge_claims_npi(all_states_claims_npis, npi_xwalk)
    claims_npi_provider = merge_provider_info(claims_npi)
    claims_npi_zip_county_cbsa_merge = merge_facility_cbsa_codes(claims_npi_provider)
    area_wage_index = merge_wage_indexes(claims_npi_zip_county_cbsa_merge, year)
    area_wage_index.to_csv(f'/{directory_to_store_wage_index}/{year}_facility_area_wage_index.csv')
    
    print(f'{year} area_wage_index read out') 

    print(len(area_wage_index['NPI'].value_counts()))
    return area_wage_index 


